# PyNite comparison workflow

This notebook mirrors the main mixed frame/truss workflow, but uses **PyNite** as the analysis backend.

The purpose is not to replace the educational stiffness-method solver immediately. The purpose is to create a second, independently implemented analysis path for comparison.

Important modelling differences:

- CSV `frame` elements become ordinary PyNite frame members.
- CSV `truss` elements become **pin-ended PyNite members** by releasing local `Ry/Rz` rotations at both ends.
- For direct solver-to-solver comparison, the recommended first pass is `SECTION_MODE = 'isotropic'`, because PyNite and the educational solver do not use exactly the same local-axis construction for asymmetric L sections.
- `CROSSING_DIAGONAL_MODE = 'continuous'` is the safest default for this PyNite comparison notebook.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Robust project-root detection.
CWD = Path.cwd().resolve()
if (CWD / 'solver.py').exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / 'solver.py').exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError('Could not find solver.py in the current directory or one level above.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

from solver import read_model, SolveOptions, solve_model, summarize_result
from helpers import (
    plot_geometry,
    plot_stress_3d,
    plot_signed_stress_3d,
    plot_deformed_shape,
    display_columns,
)
from pynite_adapter import (
    PyNiteRunOptions,
    analyze_pynite_model,
    summarize_pynite_result,
)

try:
    from reporting import (
        enrich_results_for_reporting,
        export_grouped_results_to_excel,
        region_summary_table,
        physical_member_summary_table,
    )
    REPORTING_AVAILABLE = True
except Exception as exc:
    REPORTING_AVAILABLE = False
    REPORTING_IMPORT_ERROR = exc


## Installation check

If this cell fails to import PyNite, install requirements from the repository root:

```bash
python -m pip install --user -r requirements.txt
```

The finite-element package is installed from PyPI as `PyniteFEA`, but imported in Python as `Pynite`.


In [ ]:
from Pynite import FEModel3D
print('PyNite import OK:', FEModel3D)


## User-editable analysis options

For the first comparison, keep:

```python
SECTION_MODE = 'isotropic'
CROSSING_DIAGONAL_MODE = 'continuous'
```

You can later try `SECTION_MODE = 'principal'`, but differences may include local-axis convention effects, not just solver effects.


In [ ]:
# Recommended first-pass comparison setting.
SECTION_MODE = 'isotropic'
PRINCIPAL_ORIENTATION = 'strong_inward'
CROSSING_DIAGONAL_MODE = 'continuous'

TOP_N_CRITICAL = 12
DEFORMED_SHAPE_SCALE = 100.0

# Set to True to run the in-house solver beside PyNite for a direct summary comparison.
RUN_INHOUSE_COMPARISON = True


## Main load case

Edit this list just as in the main workflow notebook.


In [ ]:
LOAD_CASE_BY_LABEL = [
    {'node_label': 'high_xneg', 'Fx': 0.0, 'Fy': 0.0, 'Fz': -13_400.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
    {'node_label': 'high_xpos', 'Fx': 0.0, 'Fy': 0.0, 'Fz': -13_400.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
]

LOAD_CASE_BY_LABEL


## Load the models

In [ ]:
models = {
    'TE5': read_model('TE5', DATA_DIR / 'TE5_nodes_aligned.csv', DATA_DIR / 'TE5_elements_with_sections.csv'),
    'Z5': read_model('Z5', DATA_DIR / 'Z5_nodes_aligned.csv', DATA_DIR / 'Z5_elements_with_sections.csv'),
}

for name, model in models.items():
    print(name, 'nodes =', len(model.nodes), 'elements =', len(model.elements))
    display(model.elements['element_type'].value_counts().rename('count').to_frame())


## Geometry check

These are still the standard matplotlib geometry plots from `helpers.py`.


In [ ]:
for name, model in models.items():
    fig, ax = plot_geometry(
        model.nodes, model.elements,
        title=f'{name} full geometry — input CSV frame + truss',
        annotate=False, annotate_elements=False, show_frame=True, show_truss=True,
    )
    plt.show()

    fig, ax = plot_geometry(
        model.nodes, model.elements,
        title=f'{name} frame elements only — input CSV',
        annotate=False, annotate_elements=True, show_frame=True, show_truss=False,
    )
    plt.show()


## Solve with PyNite

In [ ]:
pynite_options = PyNiteRunOptions(
    crossing_diagonal_mode=CROSSING_DIAGONAL_MODE,
    section_mode=SECTION_MODE,
    principal_orientation=PRINCIPAL_ORIENTATION,
)

pynite_results = {}
for name, model in models.items():
    print(f'Solving {name} with PyNite...')
    pynite_results[name] = analyze_pynite_model(
        model,
        LOAD_CASE_BY_LABEL,
        options=pynite_options,
        case_name='Main',
        combo_name='Main',
    )

pynite_summary = pd.DataFrame([summarize_pynite_result(res) for res in pynite_results.values()])
display(pynite_summary)


## Optional direct comparison against the educational solver

This is useful for checking whether the two modelling paths are in the same broad range.

Expect differences. The two backends do not represent truss-like members and local axes in exactly the same way.


In [ ]:
if RUN_INHOUSE_COMPARISON:
    inhouse_options = SolveOptions(
        crossing_diagonal_mode=CROSSING_DIAGONAL_MODE,
        section_mode=SECTION_MODE,
        principal_orientation=PRINCIPAL_ORIENTATION,
    )
    inhouse_results = {
        name: solve_model(model, LOAD_CASE_BY_LABEL, inhouse_options)
        for name, model in models.items()
    }
    inhouse_summary = pd.DataFrame([summarize_result(res) for res in inhouse_results.values()])
    inhouse_summary.insert(1, 'backend', 'in_house')

    common_cols = [c for c in pynite_summary.columns if c in inhouse_summary.columns]
    comparison = pd.concat([
        pynite_summary[common_cols],
        inhouse_summary[common_cols],
    ], ignore_index=True)
    display(comparison.sort_values(['model', 'backend']))
else:
    print('RUN_INHOUSE_COMPARISON = False')


## Result tables

In [ ]:
DISPLAY_COLS = display_columns()

for name, res in pynite_results.items():
    print(f'Critical PyNite result rows for {name}')
    mf = res['member_forces'].sort_values('sigma_max_abs_MPa', ascending=False).head(TOP_N_CRITICAL)
    display(mf[[c for c in DISPLAY_COLS if c in mf.columns]])


## Absolute stress visualization

This uses the standard matplotlib helper and the PyNite-derived result table.


In [ ]:
for name, res in pynite_results.items():
    fig, ax = plot_stress_3d(
        res['nodes'], res['elements'], res['member_forces'],
        title=f'{name} PyNite absolute stress visualization — {SECTION_MODE}, {CROSSING_DIAGONAL_MODE}',
        annotate_load_points=True, annotate_elements=False, show_frame=True, show_truss=True,
    )
    plt.show()


## Signed tension/compression visualization

Positive values are tensile; negative values are compressive.


In [ ]:
for name, res in pynite_results.items():
    fig, ax = plot_signed_stress_3d(
        res['nodes'], res['elements'], res['member_forces'],
        title=f'{name} PyNite signed stress visualization — {SECTION_MODE}, {CROSSING_DIAGONAL_MODE}',
        stress_column='sigma_extreme_signed_MPa',
        annotate_load_points=True, annotate_elements=False, show_frame=True, show_truss=True,
    )
    plt.show()


## Deformed shape

In [ ]:
for name, res in pynite_results.items():
    fig, ax = plot_deformed_shape(
        res['nodes'], res['elements'], res['displacements'],
        scale=DEFORMED_SHAPE_SCALE,
        title=f'{name} PyNite deformed shape — {SECTION_MODE}, {CROSSING_DIAGONAL_MODE}',
        show_truss=True,
    )
    plt.show()


## Optional grouped XLSX export

In [ ]:
if REPORTING_AVAILABLE:
    enriched = enrich_results_for_reporting(pynite_results)
    print('Region summary:')
    display(region_summary_table(enriched))
    print('Physical members:')
    display(physical_member_summary_table(enriched).head(40))

    out_xlsx = RESULTS_DIR / f'pynite_results_{SECTION_MODE}_{CROSSING_DIAGONAL_MODE}.xlsx'
    export_grouped_results_to_excel(enriched, out_xlsx, top_n=TOP_N_CRITICAL)
    print('Wrote:', out_xlsx)
else:
    print('reporting.py could not be imported:', REPORTING_IMPORT_ERROR)


# Z5 three-case PyNite study

The remaining cells run the three specific Z5 cases using the PyNite backend.


In [ ]:
Z5_CASES = {
    'Case_1': [
        {'node_label': 'high_xneg', 'Fx': 17000.0, 'Fy': 1500.0, 'Fz': -24200.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
        {'node_label': 'high_xpos', 'Fx': 17000.0, 'Fy': 1500.0, 'Fz': -24200.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
    ],
    'Case_2': [
        {'node_label': 'high_xneg', 'Fx': 48900.0, 'Fy': 0.0, 'Fz': -13400.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
        {'node_label': 'high_xpos', 'Fx': 48900.0, 'Fy': 0.0, 'Fz': -13400.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
    ],
    'Case_3': [
        {'node_label': 'high_xneg', 'Fx': 21600.0, 'Fy': 38000.0, 'Fz': -13400.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
        {'node_label': 'high_xpos', 'Fx': 21600.0, 'Fy': 38000.0, 'Fz': -13400.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
    ],
}


In [ ]:
z5_pynite_case_results = {}
for case_name, loads in Z5_CASES.items():
    print(f'Solving Z5 {case_name} with PyNite...')
    z5_pynite_case_results[case_name] = analyze_pynite_model(
        models['Z5'],
        loads,
        options=pynite_options,
        case_name=case_name,
        combo_name=case_name,
    )

z5_pynite_summary = pd.DataFrame([
    {'case': case_name, **summarize_pynite_result(res)}
    for case_name, res in z5_pynite_case_results.items()
])
display(z5_pynite_summary)


## Z5 PyNite case visualizations

In [ ]:
for case_name, res in z5_pynite_case_results.items():
    fig, ax = plot_signed_stress_3d(
        res['nodes'], res['elements'], res['member_forces'],
        title=f'Z5 PyNite signed stress — {case_name}',
        stress_column='sigma_extreme_signed_MPa',
        annotate_load_points=True, annotate_elements=False, show_frame=True, show_truss=True,
    )
    plt.show()

    fig, ax = plot_deformed_shape(
        res['nodes'], res['elements'], res['displacements'],
        scale=DEFORMED_SHAPE_SCALE,
        title=f'Z5 PyNite deformed shape — {case_name}',
        show_truss=True,
    )
    plt.show()


## Z5 Case 1 exceedance check under PyNite

This reproduces the earlier envelope question for the PyNite backend: where does Case 1 exceed Case 2 and/or Case 3?


In [ ]:
def case1_exceedance_table(case_results):
    metrics = [
        'sigma_max_abs_MPa',
        'max_abs_N_N',
        'sigma_axial_signed_MPa',
        'max_M_resultant_Nmm',
    ]
    base = case_results['Case_1']['member_forces'].copy()
    out_rows = []
    for metric in metrics:
        c1 = base[['element_id', 'start_label', 'end_label', 'member_role', 'section_name', metric]].rename(columns={metric: 'Case_1'})
        c2 = case_results['Case_2']['member_forces'][['element_id', metric]].rename(columns={metric: 'Case_2'})
        c3 = case_results['Case_3']['member_forces'][['element_id', metric]].rename(columns={metric: 'Case_3'})
        merged = c1.merge(c2, on='element_id').merge(c3, on='element_id')
        for _, row in merged.iterrows():
            for other in ['Case_2', 'Case_3']:
                diff = float(row['Case_1'] - row[other])
                if diff > 0:
                    pct = diff / abs(float(row[other])) * 100.0 if abs(float(row[other])) > 1e-12 else float('inf')
                    out_rows.append({
                        'metric': metric,
                        'element_id': int(row['element_id']),
                        'start_label': row['start_label'],
                        'end_label': row['end_label'],
                        'member_role': row['member_role'],
                        'section_name': row['section_name'],
                        'comparison': f'Case_1 > {other}',
                        'Case_1': float(row['Case_1']),
                        other: float(row[other]),
                        'absolute_increase': diff,
                        'percent_increase': pct,
                    })
    return pd.DataFrame(out_rows).sort_values(['metric', 'element_id', 'comparison']).reset_index(drop=True)

z5_case1_exceedances = case1_exceedance_table(z5_pynite_case_results)
display(z5_case1_exceedances)


In [ ]:
out_dir = RESULTS_DIR / 'pynite_z5_three_cases'
out_dir.mkdir(exist_ok=True)

z5_pynite_summary.to_csv(out_dir / 'summary.csv', index=False)
z5_case1_exceedances.to_csv(out_dir / 'case1_element_exceedances.csv', index=False)

for case_name, res in z5_pynite_case_results.items():
    res['displacements'].to_csv(out_dir / f'{case_name}_displacements.csv', index=False)
    res['member_forces'].to_csv(out_dir / f'{case_name}_member_forces.csv', index=False)
    res['reactions'].to_csv(out_dir / f'{case_name}_reactions.csv', index=False)

print('Wrote:', out_dir)
